# SEC filings — structured extraction & question-driven summarization (LangGraph)

An experiment on long SEC documents (10-K, S-1 prospectus) where the model only ever sees a chunk at a time. Everything is driven by **one config cell** (section 2): the task prompt, a schema field list *or* a list of questions. Nothing about fields, prompts or questions is hard-coded elsewhere.

Practices borrowed from how Anthropic documents long-document work: **document first, instructions last**, XML-tagged inputs, **quote first, then answer**, page-level citations, structured output through a forced tool call.

| Step | What happens |
|---|---|
| 1-2 | Install · config (your inputs) · clients |
| 3 | **Compile** the schema/questions into a strict tool definition |
| 4 | **Read** the PDF with Azure AI Content Understanding -> one markdown string per page |
| 5-6 | **Document map** (cheap global context for every chunk) · heading-aware **chunking** |
| 7-8 | **Map** (parallel) and **reduce** (merge records / answer questions) |
| 9 | **Verify** every quote against its page; a tool-using **reviewer** fixes or drops failures |
| 10-11 | Assemble the **LangGraph** in three arms (whole-document baseline · parallel map-reduce · sequential ledger) and **compare** |

Keys are read from `AZURE_OPENAI_KEY` / `AZURE_CU_KEY` (or prompted) — never stored here.

In [ ]:
"""Install LangGraph (graph runtime), the OpenAI SDK (AzureOpenAI client), Pydantic (schema validation), RapidFuzz (quote matching),
    pandas, and the Content Understanding SDK (pre-release)."""
%pip install --quiet langgraph openai pydantic rapidfuzz pandas azure-identity python-dotenv pypdf
# Only needed when USE_CONTENT_UNDERSTANDING = True; skip this line on a machine without Content Understanding:
%pip install --pre --quiet azure-ai-contentunderstanding

## 1. Install dependencies

In [ ]:
"""User inputs and tuning knobs (no secrets in this cell).

    MODE / TASK_PROMPT / SCHEMA / QUESTIONS: what to do; edit these.
    PDF_PATH: the filing to process (None -> the first *.pdf next to this notebook).
    USE_CONTENT_UNDERSTANDING: False reads the PDF text layer locally with pypdf (no CU key, endpoint or package; text-based PDFs only).
    CHUNK_PAGES / OVERLAP_PAGES / MAX_CHUNK_TOKENS: chunking. FUZZY_THRESHOLD: quote-match score (0-100).
    USE_REVIEW: True re-checks items that failed quote verification with a reviewer (extra tokens); False drops them. REVIEWER_MAX_STEPS: tool-loop cap per failed item. CONTEXT_BUDGET_TOKENS: the baseline arm is skipped above this.
    REASONING_EFFORT / MAP_REASONING_EFFORT: `reasoning_effort` (None to omit); the map stage has its own, lower, setting.
    MAX_CONCURRENCY: parallel chunk calls. LEDGER_WAVE: chunks per wave in the ledger arm. REQUEST_TIMEOUT: seconds per model call."""
MODE = "structured"          # "structured" | "verbose"

TASK_PROMPTS = {             # one prompt per mode; edit either. It must agree with SCHEMA / QUESTIONS.
    "structured": (
        "Extract the company's key annual financial metrics (revenue, net income, total assets, "
        "operating cash flow) for every fiscal year reported in the financial statements or MD&A."
        "Extract every debt security described in the filing (notes, bonds, debentures, term loans). One record per instrument. Fill only the fields the text states for that instrument; leave the others null."
    ),
    "verbose": (
        "Gather the passages that answer the questions. Be thorough and prefer the company's own wording."
    ),
}
TASK_PROMPT = TASK_PROMPTS[MODE]

SCHEMA = {
    "record_name": "debt_instrument",
    "key": ["instrument"],
    "fields": [
        {"name": "instrument", "type": "string", "required": True,
         "description": "Name of the security as written, e.g. '5.25% Senior Notes due 2030'"},
        {"name": "amount", "type": "number", "required": False,
         "description": "Principal amount, number only, no currency symbol or commas (e.g. 500000000, or 500 if the text says '$500 million')"},
        {"name": "amount_unit", "type": "enum", "required": False,
         "values": ["USD", "USD thousands", "USD millions", "USD billions"],
         "description": "Unit the amount is stated in"},
        {"name": "maturity_date", "type": "date", "required": False,
         "description": "Maturity date as written, ideally YYYY-MM-DD; if only a year or month is given, copy that"},
        {"name": "interest_rate", "type": "number", "required": False,
         "description": "Stated coupon / interest rate in percent, number only (5.25 for 5.25%)"},
        {"name": "yield", "type": "number", "required": False,
         "description": "Yield to maturity or yield to investors in percent, number only"},
        {"name": "price", "type": "number", "required": False,
         "description": "Offering or issue price, number only (e.g. 99.5 for 99.5% of principal)"},
    ],
}

QUESTIONS = [
    "Using the financial statements, capitalization table, dilution disclosures, and growth metrics in Anthropic's S-1, construct a bull, base, and bear investment case for the next five years. Estimate revenue, gross margins, free cash flow, and an appropriate valuation range, distinguishing reported facts from modeling assumptions."
]

PDF_PATH = r"C:\Users\Sunny\Documents\LearnCode\sec_extraction\documents\Phila-City.pdf"

USE_CONTENT_UNDERSTANDING = False   # True: Azure Content Understanding (OCR + tables; needs CU key and package). False: read a text-based PDF locally with pypdf

CHUNK_PAGES = 5
OVERLAP_PAGES = 1
MAX_CHUNK_TOKENS = 30_000
FUZZY_THRESHOLD = 90
USE_REVIEW = False                # False: skip the reviewer (about a fifth of the tokens); items that fail verification are dropped
REVIEWER_MAX_STEPS = 6
CONTEXT_BUDGET_TOKENS = 200_000
REASONING_EFFORT = "medium"      # doc map, answers, reviewer
MAP_REASONING_EFFORT = "low"     # the per-chunk extraction calls: ~2x faster than medium, quotes are verified anyway
MAX_CONCURRENCY = 16             # parallel chunk calls in the `main` arm
LEDGER_WAVE = 4                  # ledger arm: chunks per wave; a wave runs in parallel and sees what earlier waves found (1 = strictly one by one)
REQUEST_TIMEOUT = 120            # seconds before one model call is abandoned and retried

OPENAI_ENDPOINT = "https://sunny-mgihwu5s-eastus2.cognitiveservices.azure.com/"
OPENAI_API_VERSION = "2025-04-01-preview"
MODEL = "gpt-5.5"        # Azure deployment name

CU_ENDPOINT = "https://sunny-mgihwu5s-eastus2.services.ai.azure.com/"
CU_API_VERSION = "2025-11-01"
CU_ANALYZER_ID = "prebuilt-layout"

## 2. Configuration — the only cell you edit

- `MODE`: `"structured"` (extract records that follow `SCHEMA`) or `"verbose"` (answer each of `QUESTIONS`).
- `TASK_PROMPT`: your instruction. It is injected into every stage *after* the document, inside fixed scaffolding (quote-first rules) that it never replaces.
- `SCHEMA`: a plain field list. Types: `string`, `number`, `integer`, `boolean`, `date`, `enum` (with `values`). `key` lists the fields that identify a record (used to merge duplicates). `page` and `evidence_quote` are added to every record automatically.
- Tuning knobs and endpoints follow below the inputs.

## 2b. Why tools go out as JSON-schema calls

This deployment (`gpt-6-astra`, chat completions) **rejects function tools**: with `reasoning_effort` set it says "Function tools with reasoning_effort are not supported", and without it (or with `'none'`) it says `'none'` is unsupported - so tools cannot be used at all on this endpoint, while strict `response_format` JSON-schema works fine alongside reasoning. `JsonSchemaTools` wraps the client: the rest of the notebook still talks in "forced tool" and "choose a tool" terms, but on the wire it is a strict schema (a forced tool *is* the schema; a choice between tools is a `{tool, arguments_json}` envelope). Same guarantee - the model can only emit schema-valid JSON - without function calling. (The Responses API would allow real tools, but this experiment is deliberately on chat completions.)

In [ ]:
"""JsonSchemaTools: wraps the Azure client so tool requests go out as strict response_format JSON-schema calls."""
import json
from types import SimpleNamespace


class JsonSchemaTools:
    """gpt-6-astra on chat completions rejects function tools (with any reasoning_effort, and 'none' is unsupported).
    Strict `response_format` works, so a tool request is sent as one and the reply is handed back shaped like a tool call:
    a forced tool becomes the schema itself; a choice between tools becomes {"tool", "arguments_json"}."""

    def __init__(self, client):
        self._client = client
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, **kw):
        tools, choice = kw.pop("tools", None), kw.pop("tool_choice", None)
        kw.pop("parallel_tool_calls", None)
        if not tools:
            return self._client.chat.completions.create(**kw)
        fns = [t["function"] for t in tools]
        forced = choice["function"]["name"] if choice else (fns[0]["name"] if len(fns) == 1 else None)
        messages = self._flatten(kw.pop("messages"))
        if forced:
            fn = next(f for f in fns if f["name"] == forced)
            schema, name = fn["parameters"], fn["name"]
        else:
            schema = {"type": "object", "properties": {"tool": {"type": "string", "enum": [f["name"] for f in fns]},
                                                       "arguments_json": {"type": "string"}},
                      "required": ["tool", "arguments_json"], "additionalProperties": False}
            name = "tool_call"
            listing = "\n".join(f"- {f['name']}: {f.get('description', '')} Arguments schema: {json.dumps(f['parameters'])}" for f in fns)
            messages = messages + [{"role": "system", "content":
                                    "Reply with the tool to use and its arguments as a JSON string in `arguments_json`. Tools:\n" + listing}]
        kw["response_format"] = {"type": "json_schema", "json_schema": {"name": name, "strict": True, "schema": schema}}
        resp = self._client.chat.completions.create(messages=messages, **kw)
        msg = resp.choices[0].message
        return SimpleNamespace(choices=[SimpleNamespace(message=self._as_tool_call(msg, forced), finish_reason=resp.choices[0].finish_reason)],
                               usage=getattr(resp, "usage", None))

    @staticmethod
    def _flatten(messages):
        """No tool-role or tool_calls messages go out: earlier calls and their results become plain assistant/user text."""
        out = []
        for m in messages:
            if m.get("role") == "tool":
                out.append({"role": "user", "content": f"<tool_result>\n{m['content']}\n</tool_result>"})
            elif m.get("tool_calls"):
                calls = [{"tool": c["function"]["name"], "arguments_json": c["function"]["arguments"]} for c in m["tool_calls"]]
                out.append({"role": "assistant", "content": json.dumps(calls[0] if len(calls) == 1 else calls)})
            else:
                out.append(m)
        return out

    @staticmethod
    def _as_tool_call(msg, forced):
        calls = None
        try:
            if forced:
                json.loads(msg.content)
                name, args = forced, msg.content
            else:
                obj = json.loads(msg.content)
                name, args = obj["tool"], obj["arguments_json"]
            calls = [SimpleNamespace(id="call_json", type="function", function=SimpleNamespace(name=name, arguments=args))]
        except Exception:                       # empty, refused or malformed -> callers see "no tool call" and retry
            pass
        return SimpleNamespace(content=msg.content, tool_calls=calls, refusal=getattr(msg, "refusal", None))

In [ ]:
"""Create the Azure OpenAI client (`llm`) and, only when USE_CONTENT_UNDERSTANDING is True, the Content Understanding client (`cu`). Keys come from env vars, else getpass."""
import os, getpass
from pathlib import Path
from openai import AzureOpenAI
import dotenv
dotenv.load_dotenv()
OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY", "").strip() or getpass.getpass("Azure OpenAI key: ").strip()

llm = JsonSchemaTools(AzureOpenAI(azure_endpoint=OPENAI_ENDPOINT, api_version=OPENAI_API_VERSION, api_key=OPENAI_KEY,
                                  timeout=REQUEST_TIMEOUT, max_retries=2))
cu = None
if USE_CONTENT_UNDERSTANDING:
    from azure.ai.contentunderstanding import ContentUnderstandingClient
    from azure.core.credentials import AzureKeyCredential
    CU_KEY = os.environ.get("AZURE_CU_KEY", "").strip() or getpass.getpass("Azure CU key: ").strip()
    cu = ContentUnderstandingClient(endpoint=CU_ENDPOINT, credential=AzureKeyCredential(CU_KEY), api_version=CU_API_VERSION)

NB_DIR = Path.cwd()
OUT_DIR = NB_DIR / "results"
OUT_DIR.mkdir(exist_ok=True)
print("Model :", MODEL, "| mode:", MODE, "| reader:", "content_understanding" if USE_CONTENT_UNDERSTANDING else "local pypdf", "| out:", OUT_DIR)

Model : gpt-5.5 | mode: structured | reader: local pypdf | out: c:\Users\Sunny\Documents\LearnCode\sec_extraction\results


In [ ]:
"""compile_schema(schema) -> CompiledSchema: strict-mode JSON Schema + `record_findings` tool + Pydantic validator + hash.
    validate_record(c, rec) -> pydantic instance (raises on bad types/enums).  compile_questions(list[str]) -> [{"id","text"}]."""
import hashlib, json
from dataclasses import dataclass, field
from typing import Any, Optional, Literal

from pydantic import ConfigDict, create_model

_JSON_TYPES = {"string": "string", "number": "number", "integer": "integer", "boolean": "boolean", "date": "string"}
_PY_TYPES = {"string": str, "number": float, "integer": int, "boolean": bool, "date": str}


@dataclass
class CompiledSchema:
    record_name: str
    key: list
    model: Any
    json_schema: dict
    tool: dict
    hash: str
    numeric_fields: list = field(default_factory=list)   # number/integer fields: their value must appear in the evidence quote


def compile_schema(schema: dict) -> CompiledSchema:
    fields = schema.get("fields") or []
    if not fields:
        raise ValueError("SCHEMA needs at least one field")
    names = [f["name"] for f in fields]
    if len(set(names)) != len(names) or {"page", "evidence_quote"} & set(names):
        raise ValueError("field names must be unique and must not be 'page' or 'evidence_quote' (added automatically)")
    key = list(schema.get("key") or [])
    if not key or any(k not in names for k in key):
        raise ValueError(f"key {key} must be a non-empty list of declared field names {names}")

    props, pyfields = {}, {}
    for f in fields:
        t, required = f["type"], bool(f.get("required", True))
        if t == "enum":
            values = f.get("values") or []
            if not values:
                raise ValueError(f"enum field {f['name']!r} needs 'values'")
            prop = {"type": ["string", "null"] if not required else "string",
                    "enum": list(values) + ([] if required else [None])}
            py = Literal[tuple(values)]
        elif t in _JSON_TYPES:
            prop = {"type": [_JSON_TYPES[t], "null"] if not required else _JSON_TYPES[t]}
            py = _PY_TYPES[t]
        else:
            raise ValueError(f"unknown type {t!r} for field {f['name']!r}")
        prop["description"] = f.get("description", "")
        props[f["name"]] = prop
        pyfields[f["name"]] = (py, ...) if required else (Optional[py], None)

    props["page"] = {"type": "integer", "description": "Page number (the n of the <page n=..> tag) where the evidence is."}
    props["evidence_quote"] = {"type": "string", "description": "Short VERBATIM quote copied from that page that supports this record."}
    pyfields["page"] = (int, ...)
    pyfields["evidence_quote"] = (str, ...)

    json_schema = {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}
    model = create_model(schema.get("record_name", "Record"), __config__=ConfigDict(strict=True, extra="forbid"), **pyfields)
    tool = {"type": "function", "function": {
        "name": "record_findings", "strict": True,
        "description": f"Report every {schema.get('record_name', 'record')} found in the pages. Call with an empty list if there are none.",
        "parameters": {"type": "object", "properties": {"records": {"type": "array", "items": json_schema}},
                       "required": ["records"], "additionalProperties": False}}}
    digest = hashlib.sha256(json.dumps({"key": key, "schema": json_schema}, sort_keys=True).encode()).hexdigest()[:12]
    numeric = [f["name"] for f in fields if f["type"] in ("number", "integer")]
    return CompiledSchema(schema.get("record_name", "record"), key, model, json_schema, tool, digest, numeric)


def validate_record(c: CompiledSchema, rec: dict):
    return c.model.model_validate(rec)


def compile_questions(qs: list) -> list:
    clean = [q.strip() for q in qs if q and q.strip()]
    if not clean:
        raise ValueError("QUESTIONS needs at least one non-empty question")
    return [{"id": f"q{i}", "text": q} for i, q in enumerate(clean, 1)]

## 3. Compile your inputs

No model involved. The field list becomes (a) a **JSON Schema in Azure strict-mode form** (every property required, `additionalProperties: false`, optional fields nullable) wrapped in a forced tool `record_findings(records[])`, and (b) a **Pydantic model** that validates every record the model returns (strict: no silent coercion of strings). Note that the strict tool schema already forces a `number` field to be a JSON number, so the model itself converts `(1,234)` to a number: the prompt tells it parentheses mean negative, and section 7 checks that every `number`/`integer` value actually appears in the evidence quote (sign and a 1e3/1e6/1e9 scale are allowed for). `page` and `evidence_quote` are added to every record. A short hash of the compiled schema is carried through every run so the comparison arms provably used the same config.

In [ ]:
"""Compile the config above and save it (results/compiled_config.json) so every arm can show the same hash."""
import json
COMPILED = compile_schema(SCHEMA) if MODE == "structured" else None
QS = compile_questions(QUESTIONS) if MODE == "verbose" else []
(OUT_DIR / "compiled_config.json").write_text(json.dumps({
    "mode": MODE, "task_prompt": TASK_PROMPT,
    "schema_hash": COMPILED.hash if COMPILED else None,
    "json_schema": COMPILED.json_schema if COMPILED else None, "questions": QS}, indent=2), encoding="utf-8")
print("mode:", MODE, "| schema hash:", COMPILED.hash if COMPILED else "-", "| questions:", len(QS))

mode: structured | schema hash: f589eac11816 | questions: 0


In [ ]:
"""result_to_pages(result_dict) -> [{"n": 1-based page number, "text": markdown}]   (pure; tested offline)."""
import re

_PAGE_BREAK = re.compile(r"\s*<!--\s*PageBreak\s*-->\s*")


def result_to_pages(result: dict) -> list:
    contents = [c for c in (result.get("contents") or []) if c.get("markdown")]
    if not contents:
        return []
    md = "\n\n".join(c["markdown"] for c in contents)
    parts = _PAGE_BREAK.split(md)
    if len(parts) == 1 and len(contents) == 1 and len(contents[0].get("pages") or []) > 1:
        spans = [(p.get("pageNumber", i), p["spans"][0]) for i, p in enumerate(contents[0]["pages"], 1) if p.get("spans")]
        parts = [md[s["offset"]: s["offset"] + s["length"]] for _, s in spans]
    return [{"n": i, "text": t.strip()} for i, t in enumerate(parts, 1)]

## 4. Read the PDF with Content Understanding

`prebuilt-layout` returns the whole document as one markdown string. We split it into **one string per page** (on the `<!-- PageBreak -->` markers; if those are missing, on the per-page character spans in the result; if both are missing, the document is one page). Page numbers are what every record and answer cites, so they are kept 1-indexed and stable. The raw result is cached in `results/<name>_result.json`, so re-runs don't re-bill OCR.

In [ ]:
"""read_pdf(path) -> pages: analyze with Content Understanding (cached in results/) and split into pages. Runs only when USE_CONTENT_UNDERSTANDING is True."""
import json

def read_pdf(path):
    path = Path(path)
    cache = OUT_DIR / f"{path.stem}_result.json"
    if cache.exists():
        result = json.loads(cache.read_text(encoding="utf-8"))
    else:
        result = cu.begin_analyze_binary(analyzer_id=CU_ANALYZER_ID, binary_input=path.read_bytes()).result().as_dict()
        cache.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result_to_pages(result)

if USE_CONTENT_UNDERSTANDING:
    PDF = Path(PDF_PATH) if PDF_PATH else next(iter(sorted(NB_DIR.glob("*.pdf"))), None)
    assert PDF and PDF.exists(), f"Put a filing PDF next to the notebook (or set PDF_PATH); looked in {NB_DIR}"
    PAGES = read_pdf(PDF)
    PAGES_BY_N = {p["n"]: p["text"] for p in PAGES}
    print(f"{PDF.name}: {len(PAGES)} pages, {sum(len(p['text']) for p in PAGES):,} chars")
    print("--- page 1 preview ---\n" + PAGES[0]["text"][:600])
else:
    print("USE_CONTENT_UNDERSTANDING is False: this cell is skipped; the next cells read the PDF locally.")

USE_CONTENT_UNDERSTANDING is False: this cell is skipped; the next cells read the PDF locally.


## 4b. Alternative reader: a text-based PDF, read locally

Use this when Azure Content Understanding is not available (for example on an office machine): set `USE_CONTENT_UNDERSTANDING = False` in the config cell. It extracts the PDF's own **text layer** with `pypdf`, one string per page, and produces exactly the same `PAGES` as section 4, so everything after it (document map, chunking, map, reduce, verify, review) is unchanged.

Limits to know: it needs a PDF with selectable text (a scanned PDF gives a clear error; that needs OCR), and tables come out as lines of text rather than markdown or HTML tables. Quotes are still verified against this same text, so accuracy checking works the same way.

In [ ]:
"""read_pdf_local(path) -> [{"n": 1-based page number, "text": str}]: the text layer of a readable PDF via pypdf (tested offline)."""
import re
from pypdf import PdfReader


def read_pdf_local(path) -> list:
    pages = []
    for i, page in enumerate(PdfReader(str(path)).pages, 1):
        text = (page.extract_text() or "").replace("\x00", "")
        text = re.sub(r"[ \t]+\n", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        pages.append({"n": i, "text": text})
    if sum(len(p["text"]) for p in pages) < 20 * max(len(pages), 1):
        raise ValueError("This PDF has almost no text layer (scanned?). Local reading needs a text-based PDF; "
                         'set USE_CONTENT_UNDERSTANDING = True (OCR) for scans.')
    return pages

In [ ]:
"""Read the PDF locally. Runs only when USE_CONTENT_UNDERSTANDING is False; sets PDF, PAGES and PAGES_BY_N like section 4."""
if not USE_CONTENT_UNDERSTANDING:
    PDF = Path(PDF_PATH) if PDF_PATH else next(iter(sorted(NB_DIR.glob("*.pdf"))), None)
    assert PDF and PDF.exists(), f"Put a filing PDF next to the notebook (or set PDF_PATH); looked in {NB_DIR}"
    PAGES = read_pdf_local(PDF)
    PAGES_BY_N = {p["n"]: p["text"] for p in PAGES}
    print(f"{PDF.name}: {len(PAGES)} pages, {sum(len(p['text']) for p in PAGES):,} chars")
    print("--- page 1 preview ---\n" + PAGES[0]["text"][:600])
else:
    print("USE_CONTENT_UNDERSTANDING is True: this cell is skipped.")

Phila-City.pdf: 285 pages, 759,924 chars
--- page 1 preview ---
NEW ISSUE- BOOK-ENTRY ONLY RATINGS: (See “Ratings” herein)
In the opinion of Co-Bond Counsel, under existing statutes, regulations, rulings and court decisions, interest on the
Series B Notes will not be includible in gross income of the holders thereof for federal income tax purposes, assuming continuing
compliance by the School District with the requirements of the Internal Revenue Code of 1986, as amended. Under the laws of
the Commonwealth of Pennsylvania, as enacted and construed on the date hereof, interest on the Series B Notes is exempt from
Pennsylvania personal income tax and Pennsyl


In [ ]:
"""Shared plumbing: USAGE/FAILURES logs, call_llm, est_tokens, build_document_map, Chunk, chunk_pages, render_chunk."""
import re
from dataclasses import dataclass, field

USAGE: list = []       # one {"stage", "in", "out"} per model call
FAILURES: list = []    # chunks/items that failed after a retry; the run goes on


def reset_run_state():
    USAGE.clear()
    FAILURES.clear()


def usage_by_stage() -> dict:
    out = {}
    for u in USAGE:
        s = out.setdefault(u["stage"], {"in": 0, "out": 0, "calls": 0})
        s["in"] += u["in"]; s["out"] += u["out"]; s["calls"] += 1
    return out


def call_llm(llm, model, stage, messages, tools=None, tool_choice=None):
    """One chat.completions call; logs token usage under `stage`."""
    kw = {"model": model, "messages": messages}
    if tools:
        kw.update(tools=tools, parallel_tool_calls=False)
        if tool_choice:
            kw["tool_choice"] = {"type": "function", "function": {"name": tool_choice}}
    effort = MAP_REASONING_EFFORT if stage == "map" else REASONING_EFFORT
    if effort:
        kw["reasoning_effort"] = effort
    resp = llm.chat.completions.create(**kw)
    u = getattr(resp, "usage", None)
    USAGE.append({"stage": stage, "in": getattr(u, "prompt_tokens", 0) or 0, "out": getattr(u, "completion_tokens", 0) or 0})
    return resp


def est_tokens(text: str) -> int:
    return len(text) // 4 + 1          # rough, offline (no tokenizer download)


# ----------------------------------------------------------------- headings
_HEADING = re.compile(r"^\s*(?:#{1,2}\s+(?P<md>.+?)|(?P<item>Item\s+\d+[A-C]?\..*?))\s*$", re.IGNORECASE)


def page_heading(text: str) -> str:
    """Heading text if one of the first 3 non-empty lines looks like a section heading, else ''."""
    lines = [l for l in text.splitlines() if l.strip()][:3]
    for l in lines:
        m = _HEADING.match(l)
        if m:
            return (m.group("md") or m.group("item")).strip()
    return ""


def build_document_map(pages, llm, model) -> str:
    head = "\n\n".join(f"<page n=\"{p['n']}\">\n{p['text'][:2500]}\n</page>" for p in pages[:3])
    outline = [f"page {p['n']}: {h}" for p in pages if (h := page_heading(p["text"]))][:150]
    prompt = (f"<first_pages>\n{head}\n</first_pages>\n\n<headings>\n" + "\n".join(outline) + "\n</headings>\n\n"
              f"The document has {len(pages)} pages. Write a document map in under 300 words: the document type "
              "(e.g. Form 10-K / S-1), the company, the fiscal period, an outline of sections with page numbers, and any "
              "defined terms (e.g. 'the Company' = X). Use only what is shown above; say 'unknown' otherwise.")
    resp = call_llm(llm, model, "doc_map", [{"role": "user", "content": prompt}])
    return (resp.choices[0].message.content or "").strip()


# ------------------------------------------------------------------ chunking
@dataclass
class Chunk:
    id: int
    start: int
    end: int
    heading: str
    pages: list = field(default_factory=list)


def _last_line(text):
    lines = [l for l in text.splitlines() if l.strip()]
    return lines[-1].strip() if lines else ""


def _first_line(text):
    lines = [l for l in text.splitlines() if l.strip()]
    return lines[0].strip() if lines else ""


def _table_continues(a_text, b_text) -> bool:
    """Page a ends in a table and page b starts with one: markdown pipe rows, or the HTML <table> that Content Understanding emits."""
    if _last_line(a_text).startswith("|") and _first_line(b_text).startswith("|"):
        return True
    a_ends = _last_line(a_text).endswith("</table>") or a_text.count("<table") > a_text.count("</table>")
    b_starts = _first_line(b_text).startswith("<table") or b_text.count("</table>") > b_text.count("<table")
    return a_ends and b_starts


def chunk_pages(pages, size=None, overlap=None, max_tokens=None, min_section=2) -> list:
    size = size or CHUNK_PAGES
    overlap = OVERLAP_PAGES if overlap is None else overlap
    max_tokens = max_tokens or MAX_CHUNK_TOKENS
    if not pages:
        return []
    idx = {p["n"]: i for i, p in enumerate(pages)}
    # section starts: a new heading (differing from the previous one) on a page, once the section has >= min_section pages
    heading_at, last, starts = {}, "", [0]
    for i, p in enumerate(pages):
        h = page_heading(p["text"])
        if h and h != last:
            if i - starts[-1] >= min_section:
                starts.append(i)
            last = h
        heading_at[i] = last
    bounds = list(zip(starts, starts[1:] + [len(pages)]))
    chunks = []
    for a, b in bounds:                       # section = pages[a:b]
        s = a
        while True:
            e = min(s + size - 1, b - 1)
            cap = min(s + size + 1, b - 1)
            while e < cap and _table_continues(pages[e]["text"], pages[e + 1]["text"]):
                e += 1
            while e > s and est_tokens("".join(p["text"] for p in pages[s:e + 1])) > max_tokens:
                e -= 1
            chunks.append(Chunk(len(chunks), pages[s]["n"], pages[e]["n"], heading_at[s], pages[s:e + 1]))
            if e >= b - 1:
                break
            s = max(e - overlap + 1, s + 1)
    return chunks


def render_chunk(chunk, doc_map, ledger=None) -> str:
    """Document first: <document_map>, <section>, <pages>, optional <ledger>. Instructions are appended by the caller."""
    body = "\n".join(f'<page n="{p["n"]}">\n{p["text"]}\n</page>' for p in chunk.pages)
    out = f"<document_map>\n{doc_map}\n</document_map>\n<section>{chunk.heading}</section>\n<pages>\n{body}\n</pages>"
    if ledger:
        out += f"\n<ledger>\n{ledger}\n</ledger>"
    return out

## 5. Shared plumbing, document map and chunking

- `call_llm` is the single door to `chat.completions.create`: it adds `reasoning_effort`, forces/limits tools, and logs token usage per **stage** so the comparison table can show where tokens go. `FAILURES` collects chunks that failed twice; the run continues.
- **Document map**: one cheap call that reads the first pages and the list of headings, and writes a <300-word description (document type, company, period, section outline, defined terms). It is prepended to *every* chunk prompt, so parallel chunks that never see each other still know where they sit — the main answer to "each inference doesn't know what the previous one did".
- **Chunking** is structure-aware, not embedding-based: SEC filings already have Items/headings. Split at headings first (a heading that merely repeats a running page header is ignored, and sections shorter than 2 pages are not split off), then pack windows of `CHUNK_PAGES` with `OVERLAP_PAGES` overlap. A window is extended by a page rather than cut through a table, and shrunk if it would exceed `MAX_CHUNK_TOKENS`.

In [ ]:
"""Build the document map and the chunks for the loaded PDF (this is the first model call; one cheap request)."""
DOC_MAP = build_document_map(PAGES, llm, MODEL)
CHUNKS = chunk_pages(PAGES)
print(DOC_MAP, "\n")
print(f"{len(CHUNKS)} chunks:", [(c.start, c.end) for c in CHUNKS])

**Document type:** Official Statement / municipal securities offering document for a new issue, book-entry-only issuance.

**Issuer / “company”:** The School District of Philadelphia, Pennsylvania.

**Fiscal period:** Unknown. The securities are identified as **Tax and Revenue Anticipation Notes, Series B of 2026-2027**, dated the date of delivery and due **June 30, 2027**.

**Security / offering:**  
- **$591,910,000** Tax and Revenue Anticipation Notes, Series B of 2026-2027.  
- Interest stated to be federally tax-exempt under existing law, assuming compliance with the Internal Revenue Code; also exempt from Pennsylvania personal income tax and Pennsylvania corporate net income tax.  
- Fully registered, book-entry-only notes in denominations of $5,000 or whole multiples.  
- Secured by a pledge of taxes and revenues of the School District to be received from issuance through June 30, 2027, subject to qualifications shown.

**Visible outline / section map:**  
- Cover / offering sum

In [ ]:
"""build_prompt, extract_structured, extract_evidence, merge_records, answer_questions."""
import json, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
from pydantic import ValidationError

# Config errors (bad parameter, bad key, missing deployment) fail loudly; throttling/outages back off and retry.
FATAL_API_ERRORS = (openai.BadRequestError, openai.AuthenticationError, openai.PermissionDeniedError, openai.NotFoundError)
TRANSIENT_API_ERRORS = (openai.RateLimitError, openai.APIConnectionError, openai.InternalServerError)
_BACKOFF = (1, 2, 4)      # seconds between retries of a transient error

SYSTEM = ("You are a careful analyst reading excerpts of an SEC filing (10-K, S-1 prospectus). "
          "You only report what the supplied pages state.")

_RULES_COMMON = (
    "Rules:\n"
    "1. The <pages> above are your ONLY source. Never use outside knowledge, never infer or compute values that are not stated.\n"
    "2. Work quote-first: find the exact passage, then fill in the answer from that passage.\n"
    "3. Every item needs the page number (the n of its <page n=..> tag) and a short VERBATIM quote (<= 25 words) copied from that page.\n"
    "4. You only see some pages of a longer document (see <document_map>). Returning nothing is correct when the pages have nothing relevant.\n"
)
_RULES = {
    "structured": _RULES_COMMON + "5. Report by calling record_findings, once, with every record you found (an empty list is allowed).\n"
                  "5a. Numbers: copy each number exactly as printed in the quote. Parentheses mean negative: (1,234) is -1234. State the value in "
                  "the unit the page uses (e.g. 1,234 under 'in millions' is 1234, with the unit field set accordingly if there is one). "
                  "The value must appear in the quote.\n"
                  "6. If a <ledger> is present it lists what earlier pages already produced: return only NEW records, or records that "
                  "correct a ledger entry.\n",
    "verbose": _RULES_COMMON + "5. Report by calling report_evidence, once, listing quotes under the id of the question they bear on.\n"
               "6. If a <ledger> is present it lists evidence already gathered: add only NEW evidence.\n",
}

EVIDENCE_TOOL = {"type": "function", "function": {
    "name": "report_evidence", "strict": True,
    "description": "Report verbatim evidence quotes that bear on the questions. Empty list if none of the pages bear on them.",
    "parameters": {"type": "object", "properties": {"items": {"type": "array", "items": {
        "type": "object", "properties": {
            "question_id": {"type": "string"}, "page": {"type": "integer"},
            "quote": {"type": "string", "description": "Short VERBATIM quote from that page."}},
        "required": ["question_id", "page", "quote"], "additionalProperties": False}}},
        "required": ["items"], "additionalProperties": False}}}

ANSWER_TOOL = {"type": "function", "function": {
    "name": "submit_answer", "strict": True,
    "description": "Submit the answer to the question, with the page numbers it relies on.",
    "parameters": {"type": "object", "properties": {
        "text": {"type": "string"}, "citations": {"type": "array", "items": {"type": "integer"}}},
        "required": ["text", "citations"], "additionalProperties": False}}}


def build_prompt(task_prompt: str, mode: str, questions=None) -> str:
    """Fixed scaffolding, then (verbose) the questions, then the user's task LAST."""
    q = ""
    if questions:
        q = "<questions>\n" + "\n".join(f"{x['id']}: {x['text']}" for x in questions) + "\n</questions>\n\n"
    return f"{_RULES[mode]}\n{q}<task>\n{task_prompt}\n</task>"


def _tool_args(resp, name):
    msg = resp.choices[0].message
    if not msg.tool_calls:
        raise ValueError(f"no tool call (refusal={getattr(msg, 'refusal', None)!r}, content={(msg.content or '')[:80]!r})")
    call = msg.tool_calls[0]
    if call.function.name != name:
        raise ValueError(f"unexpected tool {call.function.name!r}")
    args = json.loads(call.function.arguments)
    if not isinstance(args, dict):
        raise ValueError("tool arguments are not an object")
    return args


def _call_with_backoff(llm, model, stage, messages, tool):
    for delay in (*_BACKOFF, None):
        try:
            return call_llm(llm, model, stage, messages, tools=[tool], tool_choice=tool["function"]["name"])
        except TRANSIENT_API_ERRORS:
            if delay is None:
                raise
            time.sleep(delay)


def _forced_call(llm, model, stage, system, user, tool, chunk_id):
    """Call with a forced tool; retry once with a nudge. Returns the arguments dict, or None after logging the failure.
    FATAL_API_ERRORS propagate (a rejected parameter would otherwise fail every chunk quietly)."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    err = None
    for _ in range(2):
        try:
            resp = _call_with_backoff(llm, model, stage, messages, tool)
            return _tool_args(resp, tool["function"]["name"])
        except FATAL_API_ERRORS:
            raise
        except Exception as e:                                                    # bad JSON, refusal, exhausted retries
            err = e
            messages = messages + [{"role": "user", "content":
                                    f"Your last reply was unusable ({type(e).__name__}). Call {tool['function']['name']} with valid arguments."}]
    FAILURES.append({"stage": stage, "chunk": chunk_id, "error": f"{type(err).__name__}: {err}"[:300]})
    return None


def extract_structured(chunk, doc_map, c, llm, model, task_prompt, ledger=None) -> list:
    user = render_chunk(chunk, doc_map, ledger) + "\n\n" + build_prompt(task_prompt, "structured")
    args = _forced_call(llm, model, "map", SYSTEM, user, c.tool, chunk.id)
    if args is None:
        return []
    out = []
    for r in args.get("records", []):
        try:
            d = validate_record(c, r).model_dump()
            d["_chunk"] = chunk.id
        except ValidationError as e:
            bad = ", ".join(str(x["loc"][0]) for x in e.errors() if x["loc"])
            d = {**r, "_invalid": f"invalid field(s): {bad}", "_chunk": chunk.id}
        out.append(d)
    return out


def extract_evidence(chunk, doc_map, questions, llm, model, task_prompt, ledger=None) -> dict:
    ids = {q["id"] for q in questions}
    out = {q["id"]: [] for q in questions}
    user = render_chunk(chunk, doc_map, ledger) + "\n\n" + build_prompt(task_prompt, "verbose", questions)
    args = _forced_call(llm, model, "map", SYSTEM, user, EVIDENCE_TOOL, chunk.id)
    for it in (args or {}).get("items", []):
        if it.get("question_id") in ids and isinstance(it.get("page"), int) and (it.get("quote") or "").strip():
            out[it["question_id"]].append({"page": it["page"], "quote": it["quote"].strip(), "_chunk": chunk.id})
    return out


# ------------------------------------------------------------------- reduce
def _nv(v):
    return v.strip().casefold() if isinstance(v, str) else v


def merge_records(recs, key):
    """Deterministic merge by `key`. Returns (merged, conflicts); conflicting variants are all kept with `_conflict=True`."""
    groups = {}
    for r in recs:
        groups.setdefault(tuple(_nv(r.get(k)) for k in key), []).append(r)
    merged, conflicts = [], []
    for k, grp in groups.items():
        grp = [r for r in grp if (r.get("evidence_quote") or "").strip()] or grp
        variants = {}
        for r in grp:
            sig = json.dumps({f: _nv(v) for f, v in r.items() if f not in ("page", "evidence_quote") and not f.startswith("_")},
                             sort_keys=True, default=str)
            variants.setdefault(sig, r)
        if len(variants) == 1:
            merged.append(next(iter(variants.values())))
        else:
            flagged = [{**r, "_conflict": True} for r in variants.values()]
            merged.extend(flagged)
            conflicts.append({"key": k, "records": flagged})
    return merged, conflicts


MAX_EVIDENCE_PER_CALL = 250      # quotes are <= 25 words, so this is ~10-15k tokens
ANSWER_WORKERS = 8


def _answer_once(llm, model, task_prompt, question, items):
    """items: [{"pages": [int], "text": str}] -> {"text", "citations"} (citations limited to pages present in items)."""
    ev = "\n".join(f"[pages {', '.join(map(str, it['pages']))}] {it['text']}" for it in items)
    user = (f"<evidence>\n{ev}\n</evidence>\n\n<question>\n{question['text']}\n</question>\n\n"
            "Answer using only the evidence above; cite the page numbers you rely on; say plainly where the evidence conflicts "
            f"or is incomplete.\n\n<task>\n{task_prompt}\n</task>")
    args = _forced_call(llm, model, "answer", SYSTEM, user, ANSWER_TOOL, -1)
    if args is None:
        return {"text": "(answer failed - see FAILURES)", "citations": []}
    allowed = {p for it in items for p in it["pages"]}
    return {"text": args["text"], "citations": sorted({p for p in args["citations"] if p in allowed})}


def _answer_question(q, items, llm, model, task_prompt, fanin):
    """Answer one question; evidence beyond MAX_EVIDENCE_PER_CALL is answered in parallel batches of `fanin`, then combined."""
    while len(items) > MAX_EVIDENCE_PER_CALL:
        batches = [items[i:i + fanin] for i in range(0, len(items), fanin)]
        with ThreadPoolExecutor(min(len(batches), ANSWER_WORKERS)) as pool:
            partial = list(pool.map(lambda b: _answer_once(llm, model, task_prompt, q, b), batches))
        items = [{"pages": p["citations"] or [0], "text": p["text"]} for p in partial]
    return _answer_once(llm, model, task_prompt, q, items)


def answer_questions(evidence, questions, llm, model, task_prompt, fanin=125, progress=None) -> dict:
    """One answer per question; questions run in parallel. `progress(str)` is called from this thread as each finishes."""
    say = progress or (lambda m: None)
    out, work = {}, {}
    for q in questions:
        items = [{"pages": [e["page"]], "text": e["quote"]} for e in evidence.get(q["id"], [])]
        if items:
            work[q["id"]] = (q, items)
        else:
            out[q["id"]] = {"text": "Not addressed in the document.", "citations": []}
    with ThreadPoolExecutor(max(1, len(work))) as pool:
        futures = {pool.submit(_answer_question, q, items, llm, model, task_prompt, fanin): (q["id"], len(items))
                   for q, items in work.values()}
        for qid, n in futures.values():
            say(f"  answering {qid} from {n} evidence items ...")
        for fut in as_completed(futures):
            qid, n = futures[fut]
            out[qid] = fut.result()
            say(f"  answered {qid}")
    return {q["id"]: out[q["id"]] for q in questions}

## 6. Map and reduce

**Prompt layout** (what Anthropic recommends for long inputs): the system message is a short role; the user message is `<document_map>` -> `<section>` -> `<pages>` (each `<page n="..">`) -> optional `<ledger>` -> the instructions **last**. The instructions are fixed scaffolding (quote first, verbatim evidence, never infer, empty is fine) followed by **your** `TASK_PROMPT` inside `<task>`; your prompt never replaces the scaffolding. `build_prompt` is the only place those words live, and the cell below the functions prints one assembled prompt so nothing is hidden.

**Map** — one model call per chunk, always through a *forced* tool call (`tool_choice`), so the output is structured by construction:
- structured: `record_findings(records[])` -> 0..n records, each with `page` + verbatim `evidence_quote`. Each record is checked by Pydantic; failures are kept and flagged `_invalid` (they go to the reviewer, not silently coerced).
- verbose: `report_evidence(items[])` -> for each question, the quotes (with pages) that bear on it; questions a chunk doesn't touch simply get nothing.
- A bad reply (invalid JSON, refusal, no tool call) is retried once with a nudge; after that the chunk is logged in `FAILURES` and the run continues.

**Reduce** —
- structured: deterministic code, no model. Group by the schema's `key`; drop evidence-less duplicates; identical values collapse to one record; different values for the same key are **both kept and flagged `_conflict`**.
- verbose: gather the evidence per question and have the model write one answer per question with page citations (a question with no evidence is answered "not addressed" with no model call). If a question has more than 40 evidence items, answer in batches first and combine the partial answers (hierarchical reduce).

In [ ]:
"""Show the exact instructions the model gets for every chunk (scaffolding + your task last), so nothing is hidden."""
print(build_prompt(TASK_PROMPT, MODE, QS if MODE == "verbose" else None))

Rules:
1. The <pages> above are your ONLY source. Never use outside knowledge, never infer or compute values that are not stated.
2. Work quote-first: find the exact passage, then fill in the answer from that passage.
3. Every item needs the page number (the n of its <page n=..> tag) and a short VERBATIM quote (<= 25 words) copied from that page.
4. You only see some pages of a longer document (see <document_map>). Returning nothing is correct when the pages have nothing relevant.
5. Report by calling record_findings, once, with every record you found (an empty list is allowed).
5a. Numbers: copy each number exactly as printed in the quote. Parentheses mean negative: (1,234) is -1234. State the value in the unit the page uses (e.g. 1,234 under 'in millions' is 1234, with the unit field set accordingly if there is one). The value must appear in the quote.
6. If a <ledger> is present it lists what earlier pages already produced: return only NEW records, or records that correct a ledger e

In [ ]:
"""verify_quote, verify_all, review: deterministic quote checking plus a capped, tool-using reviewer for failures."""
from concurrent.futures import ThreadPoolExecutor
import json, re
from rapidfuzz import fuzz

_TAGS = re.compile(r"<[^>]+>")
_MARKS = re.compile(r"[|*#_`>\\]")


def _norm(s: str) -> str:
    return " ".join(_MARKS.sub(" ", _TAGS.sub(" ", s or "")).casefold().split())


MIN_QUOTE_CHARS = 6
# a number not glued to letters (so OCR noise like 'mi1lion' is not read as a number)
_NUM = re.compile(r"(?<![A-Za-z\d])\d[\d,]*(?:\.\d+)?")


def _numbers(norm_text: str) -> list:
    out = []
    for m in _NUM.findall(norm_text):
        try:
            out.append(float(m.replace(",", "").rstrip(".")))
        except ValueError:
            pass
    return out


def verify_quote(quote, pages_by_n, page, threshold=None) -> bool:
    """True if `quote` fuzzy-matches the text of `page` (or `page` + the next page) AND every number in the quote
    appears there exactly (fuzzy matching alone lets a changed digit through: 97% similar)."""
    threshold = FUZZY_THRESHOLD if threshold is None else threshold
    nq = _norm(quote)
    if len(nq) < MIN_QUOTE_CHARS or page not in pages_by_n:
        return False
    hay = _norm(pages_by_n[page])
    if page + 1 in pages_by_n:
        hay = hay + " " + _norm(pages_by_n[page + 1])
    if not set(_numbers(nq)) <= set(_numbers(hay)):
        return False
    return fuzz.partial_ratio(nq, hay) >= threshold


def value_in_quote(value, quote) -> bool:
    """A numeric field's value must appear in its quote (sign ignored: parentheses mean negative; scale of 1e3/1e6/1e9 allowed)."""
    if value is None:
        return True
    v = abs(float(value))
    for n in _numbers(_norm(quote)):
        for s in (1.0, 1e3, 1e6, 1e9):
            if abs(n * s - v) <= 1e-6 * max(1.0, v) or abs(n - v * s) <= 1e-6 * max(1.0, n):
                return True
    return False


def verify_all(items, pages_by_n, quote_key="evidence_quote", numeric_fields=()):
    """Split items into (verified, failed). Items already flagged `_invalid` always fail; numeric fields must appear in the quote."""
    ok, bad = [], []
    for it in items:
        passed = ("_invalid" not in it and verify_quote(it.get(quote_key), pages_by_n, it.get("page"))
                  and all(value_in_quote(it.get(f), it.get(quote_key)) for f in numeric_fields))
        (ok if passed else bad).append(it)
    return ok, bad


GET_PAGES_TOOL = {"type": "function", "function": {
    "name": "get_pages", "description": "Read the source markdown of pages start..end (at most 3 pages).",
    "parameters": {"type": "object", "properties": {"start": {"type": "integer"}, "end": {"type": "integer"}},
                   "required": ["start", "end"]}}}
SUBMIT_REVIEW_TOOL = {"type": "function", "function": {
    "name": "submit_review", "description": "Finish the review of this item.",
    "parameters": {"type": "object", "properties": {
        "action": {"type": "string", "enum": ["fixed", "dropped"]},
        "record_json": {"type": "string", "description": "When fixed: the corrected item as JSON with the same fields "
                                                         "(page and a verbatim quote that really is on that page). Else ''."}},
        "required": ["action", "record_json"]}}}
REVIEW_SYSTEM = ("You review one extracted item from an SEC filing against the source pages. Use get_pages to read the pages, then call "
                 "submit_review. Fix the item only if the pages clearly support a correct version (value and a quote copied "
                 "verbatim from the page); otherwise drop it. Never invent anything.")


def _pages_text(pages_by_n, start, end):
    start, end = int(start), min(int(end), int(start) + 2)
    return "\n".join(f'<page n="{n}">\n{pages_by_n[n]}\n</page>' for n in range(start, end + 1) if n in pages_by_n) or "(no such pages)"


REVIEW_WORKERS = 8      # failed items reviewed at the same time


def review(failed, pages_by_n, llm, model, max_steps=None, quote_key="evidence_quote", validator=None, task_prompt="",
           numeric_fields=()) -> list:
    """Reviewer loop per failed item (items run in parallel) -> [{"item", "action": "fixed"|"dropped", "record": dict|None, "reason"}]."""
    max_steps = REVIEWER_MAX_STEPS if max_steps is None else max_steps
    def one(item):
        shown = {k: v for k, v in item.items() if not k.startswith("_") or k == "_invalid"}
        why = f"Pydantic rejected it: {item['_invalid']}" if "_invalid" in item else "its quote was not found on the cited page"
        messages = [{"role": "system", "content": REVIEW_SYSTEM},
                    {"role": "user", "content": f"<task>\n{task_prompt}\n</task>\n<item>\n{json.dumps(shown, default=str)}\n</item>\n"
                                                f"This item was flagged because {why}."}]
        outcome = {"item": item, "action": "dropped", "record": None, "reason": f"no decision within {max_steps} steps"}
        for _ in range(max_steps):
            try:
                msg = call_llm(llm, model, "review", list(messages), tools=[GET_PAGES_TOOL, SUBMIT_REVIEW_TOOL]).choices[0].message
            except Exception as e:                      # one flaky call must not lose the whole arm's map results
                outcome["reason"] = f"reviewer call failed: {type(e).__name__}"
                FAILURES.append({"stage": "review", "chunk": -1, "error": f"{type(e).__name__}: {e}"[:300]})
                break
            if not msg.tool_calls:
                outcome["reason"] = "reviewer made no tool call"
                break
            tc = msg.tool_calls[0]
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                outcome["reason"] = "reviewer sent invalid JSON"
                break
            if tc.function.name == "submit_review":
                outcome = _finish_review(item, args, pages_by_n, quote_key, validator, numeric_fields)
                break
            messages += [{"role": "assistant", "content": msg.content, "tool_calls": [
                            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}]},
                         {"role": "tool", "tool_call_id": tc.id, "content": _pages_text(pages_by_n, args.get("start", 1), args.get("end", 1))}]
        return outcome

    if not failed:
        return []
    with ThreadPoolExecutor(min(len(failed), REVIEW_WORKERS)) as pool:     # items are independent; order is kept
        return list(pool.map(one, failed))


def _finish_review(item, args, pages_by_n, quote_key, validator, numeric_fields=()):
    drop = lambda why: {"item": item, "action": "dropped", "record": None, "reason": why}
    if args.get("action") != "fixed":
        return drop("reviewer dropped it")
    try:
        rec = json.loads(args.get("record_json") or "")
        if validator:
            rec = validator(rec)
    except Exception as e:
        return drop(f"fix rejected: {type(e).__name__}")
    if not verify_quote(rec.get(quote_key), pages_by_n, rec.get("page")) or not all(value_in_quote(rec.get(f), rec.get(quote_key)) for f in numeric_fields):
        return drop("fixed quote still not found on the page (or its value is not in the quote)")
    return {"item": item, "action": "fixed", "record": rec, "reason": "fixed by reviewer"}

## 7. Verify the evidence, then review what fails

Claude has a Citations API that ties answers to source sentences; Azure OpenAI chat completions doesn't, so we build the equivalent: every record / evidence item carries a **verbatim quote and a page**, and `verify_quote` checks the quote against the page text.

- **Deterministic and free** — no model call. Both sides are normalised (HTML tags, table pipes and markdown marks removed, case and whitespace collapsed) and compared with RapidFuzz `partial_ratio >= FUZZY_THRESHOLD`, which tolerates OCR noise (`l` vs `1`).
- The quote is searched on the cited page **plus the next page**, so a quote that straddles a page break, or a model that is off by one page, still verifies. A fabricated quote, an empty quote or a non-existent page fails.
- Items that fail (and records Pydantic already rejected) go to a **reviewer**: a short tool loop where the model can call `get_pages(start, end)` to read the source, then `submit_review(action, record_json)` with `fixed` or `dropped`. A "fix" must itself pass quote verification (and schema validation) or it is dropped. The loop is capped at `REVIEWER_MAX_STEPS` calls per item, and a reply with no tool call counts as a drop. So the only model cost in this stage is for items that failed.

In [ ]:
"""RunConfig, build_graph(arm, cfg), run_arm(arm, cfg) -> RunResult, comparison_df(runs)."""
import hashlib, json, operator, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Annotated, Any, TypedDict

import pandas as pd
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

ARMS = ("baseline", "main", "ledger")


@dataclass
class RunConfig:
    mode: str
    task_prompt: str
    compiled: Any            # CompiledSchema (structured) or None
    questions: list          # compile_questions output (verbose) or []
    llm: Any
    model: str
    pages: list
    max_concurrency: int = 4
    on_progress: Any = None      # callable(str); default prints
    ledger_wave: int = 1         # ledger arm: chunks per wave (1 = strictly sequential)
    use_review: bool = True      # False: failed items are dropped without a reviewer call


class State(TypedDict, total=False):
    doc_map: str
    chunks: list
    raw: Annotated[list, operator.add]     # per-chunk map outputs (records, or evidence dicts)
    items: Any
    invalid: list
    conflicts: list
    verified: list
    failed: list
    reviewed: list
    final: Any
    skipped: str
    i: int                                   # ledger arm: index of the next chunk


@dataclass
class RunResult:
    arm: str
    results: Any
    usage: dict
    seconds: float
    tokens_in: int
    tokens_out: int
    verified_rate: Any
    conflicts: int
    failures: int
    skipped: str
    config_hash: str
    n_items: int


def config_hash(cfg: RunConfig) -> str:
    if cfg.compiled:
        return cfg.compiled.hash
    return hashlib.sha256(json.dumps([cfg.questions, cfg.task_prompt]).encode()).hexdigest()[:12]


def _evidence_flat(by_q):
    return [{"qid": q, **e} for q, es in by_q.items() for e in es]


def _ledger_text(cfg, raw):
    if cfg.mode == "structured":
        valid = [r for r in raw if "_invalid" not in r]
        merged, _ = merge_records(valid, cfg.compiled.key)
        skip = ("evidence_quote", "page")
        return "\n".join("- " + "; ".join(f"{k}={v}" for k, v in r.items() if not k.startswith("_") and k not in skip)
                         + f" (p{r['page']})" for r in merged[-200:])
    lines = []
    for ev in raw:
        for q, es in ev.items():
            lines += [f"{q} p{e['page']}: {e['quote'][:120]}" for e in es]
    return "\n".join(lines[-150:])


def build_graph(arm: str, cfg: RunConfig):
    if arm not in ARMS:
        raise ValueError(f"unknown arm {arm!r}; choose from {ARMS}")
    pages_by_n = {p["n"]: p["text"] for p in cfg.pages}
    structured = cfg.mode == "structured"

    def map_one(chunk, doc_map, ledger=None):
        if structured:
            return extract_structured(chunk, doc_map, cfg.compiled, cfg.llm, cfg.model, cfg.task_prompt, ledger)
        return [extract_evidence(chunk, doc_map, cfg.questions, cfg.llm, cfg.model, cfg.task_prompt, ledger)]

    # ---- map-stage nodes (the only part that differs between arms)
    def doc_map_node(state):
        return {"doc_map": build_document_map(cfg.pages, cfg.llm, cfg.model)}

    def chunk_node(state):
        return {"chunks": chunk_pages(cfg.pages)}

    def fan_out(state):
        return [Send("map_chunk", {"chunk": c, "doc_map": state["doc_map"]}) for c in state["chunks"]]

    def map_chunk(payload):
        return {"raw": map_one(payload["chunk"], payload["doc_map"])}

    def ledger_step(state):
        """One wave: `ledger_wave` chunks in parallel, all seeing the ledger built from the waves before."""
        i = state.get("i", 0)
        wave = state["chunks"][i:i + max(1, cfg.ledger_wave)]
        ledger = _ledger_text(cfg, state.get("raw", [])) or None
        if len(wave) == 1:
            outs = [map_one(wave[0], state["doc_map"], ledger=ledger)]
        else:
            with ThreadPoolExecutor(len(wave)) as pool:
                outs = list(pool.map(lambda c: map_one(c, state["doc_map"], ledger=ledger), wave))
        return {"raw": [r for out in outs for r in out], "i": i + len(wave)}

    def more_chunks(state):
        return "ledger_step" if state.get("i", 0) < len(state.get("chunks", [])) else "reduce"

    def whole_doc(state):
        whole = Chunk(0, cfg.pages[0]["n"], cfg.pages[-1]["n"], "", cfg.pages)
        doc_map = "The complete document is supplied below."
        need = est_tokens(render_chunk(whole, doc_map))
        if need > CONTEXT_BUDGET_TOKENS:
            return {"skipped": f"whole document (~{need:,} tokens) exceeds CONTEXT_BUDGET_TOKENS ({CONTEXT_BUDGET_TOKENS:,}); arm skipped", "raw": []}
        return {"raw": map_one(whole, doc_map)}

    # ---- shared tail
    def reduce_node(state):
        raw = state.get("raw", [])
        if structured:
            invalid = [r for r in raw if "_invalid" in r]
            merged, conflicts = merge_records([r for r in raw if "_invalid" not in r], cfg.compiled.key)
            return {"items": merged, "invalid": invalid, "conflicts": conflicts}
        by_q = {q["id"]: [] for q in cfg.questions}
        seen = set()
        for ev in raw:
            for q, es in ev.items():
                for e in es:
                    sig = (q, e["page"], " ".join(e["quote"].casefold().split()))
                    if sig not in seen:
                        seen.add(sig)
                        by_q[q].append({"page": e["page"], "quote": e["quote"]})
        return {"items": by_q, "invalid": [], "conflicts": []}

    def verify_node(state):
        if structured:
            ok, bad = verify_all(state["items"], pages_by_n, numeric_fields=cfg.compiled.numeric_fields)
            return {"verified": ok, "failed": bad + state["invalid"]}
        ok, bad = verify_all(_evidence_flat(state["items"]), pages_by_n, quote_key="quote")
        return {"verified": ok, "failed": bad}

    def validator(rec):
        if structured:
            return validate_record(cfg.compiled, rec).model_dump()
        if rec.get("qid") not in {q["id"] for q in cfg.questions} or not isinstance(rec.get("page"), int):
            raise ValueError("bad evidence record")
        return rec

    def review_node(state):
        failed = state.get("failed", [])
        if not failed or not cfg.use_review:
            return {"reviewed": []}
        return {"reviewed": review(failed, pages_by_n, cfg.llm, cfg.model, quote_key="evidence_quote" if structured else "quote",
                                   validator=validator, task_prompt=cfg.task_prompt,
                                   numeric_fields=cfg.compiled.numeric_fields if structured else ())}

    def finalize_node(state):
        fixed = [r["record"] for r in state.get("reviewed", []) if r["action"] == "fixed"]
        kept = state.get("verified", []) + fixed
        if structured:
            final, conflicts = merge_records(kept, cfg.compiled.key)
            return {"final": final, "conflicts": conflicts}
        evidence = {q["id"]: [] for q in cfg.questions}
        for e in kept:
            evidence[e["qid"]].append({"page": e["page"], "quote": e["quote"]})
        answers = answer_questions(evidence, cfg.questions, cfg.llm, cfg.model, cfg.task_prompt,
                                   progress=cfg.on_progress or (lambda m: print(m, flush=True)))
        return {"final": {"answers": answers, "evidence": evidence}}

    g = StateGraph(State)
    for name, fn in [("reduce", reduce_node), ("verify", verify_node), ("review", review_node), ("finalize", finalize_node)]:
        g.add_node(name, fn)
    g.add_edge("reduce", "verify"); g.add_edge("verify", "review"); g.add_edge("review", "finalize"); g.add_edge("finalize", END)

    if arm == "baseline":
        g.add_node("whole_doc", whole_doc)
        g.add_edge(START, "whole_doc")
        g.add_conditional_edges("whole_doc", lambda s: END if s.get("skipped") else "reduce", ["reduce", END])
    else:
        g.add_node("doc_map", doc_map_node); g.add_node("chunk", chunk_node)
        g.add_edge(START, "doc_map"); g.add_edge("doc_map", "chunk")
        if arm == "main":
            g.add_node("map_chunk", map_chunk)
            g.add_conditional_edges("chunk", fan_out, ["map_chunk"])
            g.add_edge("map_chunk", "reduce")
        else:
            g.add_node("ledger_step", ledger_step)
            g.add_conditional_edges("chunk", more_chunks, ["ledger_step", "reduce"])
            g.add_conditional_edges("ledger_step", more_chunks, ["ledger_step", "reduce"])
    return g.compile()


def run_arm(arm: str, cfg: RunConfig) -> RunResult:
    reset_run_state()
    t0 = time.time()
    say = cfg.on_progress or (lambda m: print(m, flush=True))
    state, done, n_chunks = {}, 0, 0
    # stream (not invoke) so progress is reported from this thread as each step finishes
    for kind, payload in build_graph(arm, cfg).stream({"raw": []}, stream_mode=["updates", "values"],
                                                      config={"max_concurrency": cfg.max_concurrency,
                                                              "recursion_limit": 2 * len(cfg.pages) + 50}):
        if kind == "values":
            state = payload
            continue
        for node, upd in payload.items():
            upd = upd or {}
            if node == "doc_map":
                say("  document map built")
            elif node == "chunk":
                n_chunks = len(upd.get("chunks", []))
                say(f"  {n_chunks} chunks")
            elif node in ("map_chunk", "ledger_step"):
                done = upd["i"] if node == "ledger_step" else done + 1
                say(f"  map {done}/{n_chunks} ({round(time.time() - t0)}s)")
            elif node == "whole_doc":
                say("  whole-document call done")
            elif node == "verify":
                say(f"  verify: {len(upd.get('verified', []))} ok, {len(upd.get('failed', []))} to review")
            elif node in ("reduce", "review", "finalize"):
                say("  review skipped (USE_REVIEW = False)" if node == "review" and not cfg.use_review else f"  {node} done")
    seconds = round(time.time() - t0, 2)
    n_map = len(state.get("chunks") or []) or 1
    map_failures = sum(1 for f in FAILURES if f["stage"] == "map")
    if not state.get("skipped") and map_failures >= n_map:
        raise RuntimeError(f"every map call failed ({map_failures}/{n_map}); first error: {FAILURES[0]['error']}")
    usage = usage_by_stage()
    failed, verified = state.get("failed", []), state.get("verified", [])
    rate = round(len(verified) / (len(verified) + len(failed)), 3) if (verified or failed) else None
    final = state.get("final", [] if cfg.mode == "structured" else {})
    if cfg.mode == "structured":
        n = len(final)
    else:
        n = sum(1 for a in final.get("answers", {}).values() if a["citations"])
    return RunResult(arm, final, usage, seconds, sum(u["in"] for u in usage.values()), sum(u["out"] for u in usage.values()),
                     rate, len(state.get("conflicts", [])), len(FAILURES), state.get("skipped", ""), config_hash(cfg), n)


def comparison_df(runs) -> pd.DataFrame:
    return pd.DataFrame([{
        "arm": r.arm, "skipped": r.skipped or "", "seconds": r.seconds, "tokens_in": r.tokens_in, "tokens_out": r.tokens_out,
        "tokens_total": r.tokens_in + r.tokens_out,
        "model_calls": sum(u["calls"] for u in r.usage.values()), "items": r.n_items, "verified_rate": r.verified_rate,
        "conflicts": r.conflicts, "failures": r.failures, "config_hash": r.config_hash} for r in runs])


def usage_table(runs) -> pd.DataFrame:
    """Token use per arm and stage (doc_map, map, review, answer), with a TOTAL row per arm."""
    rows = []
    for r in runs:
        for stage, u in r.usage.items():
            rows.append({"arm": r.arm, "stage": stage, "calls": u["calls"], "tokens_in": u["in"], "tokens_out": u["out"],
                         "tokens_total": u["in"] + u["out"]})
        rows.append({"arm": r.arm, "stage": "TOTAL", "calls": sum(u["calls"] for u in r.usage.values()), "tokens_in": r.tokens_in,
                     "tokens_out": r.tokens_out, "tokens_total": r.tokens_in + r.tokens_out})
    return pd.DataFrame(rows, columns=["arm", "stage", "calls", "tokens_in", "tokens_out", "tokens_total"])


## 8. The LangGraph — three arms on one compiled config

Every arm ends with the same three nodes (**reduce -> verify -> review -> finalize**); they differ only in how the map stage reads the document:

| Arm | Map stage | Trade-off |
|---|---|---|
| `baseline` | **One call with the whole document** in context (skipped with a message if it exceeds `CONTEXT_BUDGET_TOKENS`) | Sees everything, no stitching needed — but cost and attention degrade on long inputs |
| `main` | **Parallel map-reduce**: document map -> chunks -> `Send` fan-out, every chunk prompt carries the document map | Fast and cheap per call; chunks don't know each other, so duplicates/conflicts are resolved in the reduce step |
| `ledger` | **Sequential**: chunk N sees a compact structured *ledger* of what chunks 1..N-1 found and returns only new items | Chunks know what came before — but no parallelism, and the ledger grows |

`run_arm` clears the token/failure logs, runs the graph, and returns a `RunResult` (items, per-stage token usage, seconds, first-pass `verified_rate`, conflicts, failures, and the config hash — identical across arms because they share the compiled config). `comparison_df` puts the arms side by side.

`verified_rate` is the share of items whose quote verified **before** the reviewer touched them (a measure of how honest the first pass was); the final results contain only items that verified or were fixed.

In [ ]:
"""Run each arm on the same config and show the side-by-side comparison table."""
from IPython.display import display

cfg = RunConfig(mode=MODE, task_prompt=TASK_PROMPT, compiled=COMPILED, questions=QS, llm=llm, model=MODEL, pages=PAGES,
                max_concurrency=MAX_CONCURRENCY, ledger_wave=LEDGER_WAVE, use_review=USE_REVIEW)

# "main"    parallel map-reduce: the one to start with (minutes).
# "ledger"  waves of LEDGER_WAVE chunks; each wave sees what the earlier waves found (LEDGER_WAVE = 1 is strictly one by one, ~chunks x 10-15 s).
# "baseline" whole document in one call: skipped automatically above CONTEXT_BUDGET_TOKENS.
ARMS_TO_RUN = ["ledger"]

RUNS = {}
for arm in ARMS_TO_RUN:
    print(f"running {arm} ...", flush=True)
    RUNS[arm] = run_arm(arm, cfg)
    r = RUNS[arm]
    print(r.skipped or f"{arm}: {r.seconds}s, {r.n_items} items, verified_rate={r.verified_rate}, failures={r.failures}, "
                       f"tokens in/out/total = {r.tokens_in:,} / {r.tokens_out:,} / {r.tokens_in + r.tokens_out:,}")
print("Tokens by stage:")
display(usage_table(list(RUNS.values())))
comparison_df(list(RUNS.values()))

running ledger ...
  document map built
  71 chunks
  map 4/71 (11s)
  map 8/71 (17s)
  map 12/71 (24s)
  map 16/71 (29s)
  map 20/71 (31s)
  map 24/71 (34s)
  map 28/71 (39s)
  map 32/71 (44s)
  map 36/71 (47s)
  map 40/71 (55s)
  map 44/71 (69s)
  map 48/71 (73s)
  map 52/71 (75s)
  map 56/71 (91s)
  map 60/71 (100s)
  map 64/71 (108s)
  map 68/71 (117s)
  map 71/71 (129s)
  reduce done
  verify: 85 ok, 9 to review
  review skipped (USE_REVIEW = False)
  finalize done
ledger: 129.09s, 85 items, verified_rate=0.904, failures=0, tokens in/out/total = 450,712 / 20,784 / 471,496
Tokens by stage:


,arm,stage,calls,tokens_in,tokens_out,tokens_total
0,ledger,doc_map,1,1459,780,2239
1,ledger,map,71,449253,20004,469257
2,ledger,TOTAL,72,450712,20784,471496


,arm,skipped,seconds,tokens_in,tokens_out,tokens_total,model_calls,items,verified_rate,conflicts,failures,config_hash
0,ledger,,129.09,450712,20784,471496,72,85,0.904,3,0,f589eac11816


## 9. Run the arms and compare

Runs `baseline`, `main` and `ledger` on the same compiled config and PDF. Each run clears the usage log first, so the table shows that arm's own tokens, time, model calls, items found, first-pass `verified_rate`, conflicts and failed chunks. The `config_hash` column should be identical on every row. Edit `ARMS_TO_RUN` to run fewer.

In [ ]:
"""Show one arm's results (change SHOW_ARM) and save every arm's results to results/."""
import json
import pandas as pd
from IPython.display import display, Markdown

SHOW_ARM = "ledger"
for a, r in RUNS.items():
    (OUT_DIR / f"{PDF.stem}_{a}.json").write_text(json.dumps(
        {"arm": a, "config_hash": r.config_hash, "results": r.results, "usage": r.usage, "failures": FAILURES if a == SHOW_ARM else None},
        indent=2, default=str), encoding="utf-8")

run = RUNS.get(SHOW_ARM)
if run is None or run.skipped:
    print(SHOW_ARM, "->", run.skipped if run else "not run")
elif MODE == "structured":
    df = pd.DataFrame(run.results)
    display(df[[c for c in df.columns if not c.startswith("_") or c == "_conflict"]])
else:
    for q in QS:
        a = run.results["answers"][q["id"]]
        ev = run.results["evidence"][q["id"]]
        display(Markdown(f"### {q['text']}\n\n{a['text']}\n\n*Cited pages: {a['citations']} - {len(ev)} evidence quotes*"))
if FAILURES:
    print("\nFAILURES (last run):", *FAILURES, sep="\n  ")

,instrument,amount,amount_unit,maturity_date,interest_rate,yield,price,page,evidence_quote,_conflict
0,"Tax and Revenue Anticipation Notes, Series B o...",591910000.0,USD,"June 30, 2027",5.0,2.9,101.703,1,"$591,910,000 June 30, 2027 5.000% 2.900% 101.703",True
1,"Tax and Revenue Anticipation Notes, Series B o...",591910000.0,USD,None,NaN,NaN,NaN,5,"$591,910,000 principal amount of its Tax and R...",True
2,"Tax and Revenue Anticipation Notes, Series A o...",200000000.0,USD,None,NaN,NaN,NaN,6,"refund the entire $200,000,000 outstanding amo...",True
3,"Tax and Revenue Anticipation Notes, Series A o...",600000000.0,USD,None,NaN,NaN,NaN,6,"Tax and Revenue Anticipation Notes, Series A o...",True
4,School Lease Revenue Refunding Bonds (The Scho...,445000000.0,USD,None,NaN,NaN,NaN,7,"approximately $445,000,000 School Lease Revenu...",NaN
...,...,...,...,...,...,...,...,...,...,...
80,SERIES B (B023) General Obligation Bonds (GOB)...,49800000.0,USD,"SEPT 1, 2025",NaN,NaN,NaN,216,"November 16, 2023 49,800,000 1,070,000 SEPT 1,...",NaN
81,Gross General Obligation Bonds,2034965.0,USD thousands,None,NaN,NaN,NaN,242,"2025 2,034,965 291,280 5,090 760,660 3,091,995",NaN
82,QZAB & QSCB Bonds,291280.0,USD thousands,None,NaN,NaN,NaN,242,"2025 2,034,965 291,280 5,090 760,660 3,091,995",NaN
83,Deficit and Termination and Stadium Bonds,5090.0,USD thousands,None,NaN,NaN,NaN,242,"2025 2,034,965 291,280 5,090 760,660 3,091,995",NaN


## 10. The results (with page citations)

Structured mode: one row per record, with the page and the verbatim quote that supports it; `_conflict` marks records that share a key but disagree. Verbose mode: one answer per question with its cited pages and the evidence quotes behind it. The full run is saved to `results/<name>_<arm>.json`.

## 11. What this shows, and what frontier labs do

**Techniques in this notebook and where they come from**

| Technique | Source |
|---|---|
| Document first, instructions last; XML-tagged `<document_map>` / `<pages>` / `<task>` | Anthropic long-context prompting guidance (documented) |
| Quote first, then answer; verbatim evidence with page numbers | Anthropic long-context guidance (documented); the quote check replaces Claude's Citations API, which Azure chat completions doesn't have |
| Whole document in context when it fits (the `baseline` arm) | How claude.ai/API PDF support works: extracted text + page images, no chunking (documented) |
| Retrieval only when knowledge outgrows the window | claude.ai Projects switch to RAG automatically (documented); ChatGPT `file_search` retrieves chunks by default (documented in the API). **Not built here** - needs an embeddings deployment, and top-k can't tell you when it has found *all* the records |
| Forced tool call for structured output; deterministic merge in code | Common agent practice; not a published lab recipe |
| Global document map on every chunk; sequential ledger; hierarchical reduce; reviewer with `get_pages` | Standard map-reduce / refine patterns - **my design**, not something any lab documents as its internal method |

**Things to try on a real filing**
- Change `CHUNK_PAGES` to 2 and to 10 and compare `verified_rate` and `conflicts`.
- Compare `main` vs `ledger` on a question whose answer is spread over many sections: does the ledger reduce duplicates or conflicts, and what does it cost in seconds?
- If `baseline` is skipped, the filing is longer than `CONTEXT_BUDGET_TOKENS`; raise it only if your deployment really has the window.
- Switch `MODE` to `"verbose"` and re-run from section 3 (the compile cell). The same PDF, document map and chunks are reused.